# pydantic格式的使用

## 1、基本使用

举例1：

In [66]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os


# 1、提供大模型
load_dotenv(override=True)
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

model_with_qwen = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    # temperature=1.5,
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL,
    extra_body={"enable_thinking": False},
    # max_tokens=10,
)

In [40]:

from pydantic import BaseModel, Field


class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")
    occupation: str = Field(description="职业")

structured_model = model_with_qwen.with_structured_output(Person)

result = structured_model.invoke([
        ("system", "你只能输出符合Person模型的JSON，不要思考过程、不要多余文字，禁止输出tool_use之类内部字段"),
        ("human", "张三是一名30岁的软件工程师")
])

print(result)
print(type(result))

name='张三' age=30 occupation='软件工程师'
<class '__main__.Person'>


In [17]:
print(f"姓名：{result.name}，年龄：{result.age}岁，职业：{result.occupation}")

姓名：张三，年龄：30岁，职业：软件工程师


举例2：

In [18]:

class MovieModel(BaseModel):
    """电影的详细信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="发行年份")
    director: str = Field(description="电影导演")
    rating: float = Field(description="电影评分，满分十分")

structured_model = model_with_qwen.with_structured_output(MovieModel)


result = structured_model.invoke([
        ("system", "你只能输出符合MovieModel模型的JSON，不要思考过程、不要多余文字，禁止输出tool_use之类内部字段"),
        ("human", "给出电影盗梦空间的信息")
])

print(result)

title='盗梦空间' year=2010 director='克里斯托弗·诺兰' rating=9.4


举例3：

In [19]:
from pydantic import BaseModel, Field
from typing import List, Optional, Literal


# 定义输出结构
class SentimentAnalysis(BaseModel):
    """情感分析结果"""
    sentiment: str = Field(description="情感倾向：positive/negative/neutral")
    confidence: float = Field(description="置信度，0‑1之间")
    keywords: List[str] = Field(description="关键词列表")

# ✅ v1.x：使用 with_structured_output
structured_model = model_with_qwen.with_structured_output(SentimentAnalysis)

# 调用
text = "这个课程内容很实用，学到了很多知识，强烈推荐！"
result = structured_model.invoke(
    f"分析以下文本的情感：\n{text}"
)

print(f"类型: {type(result)}")  # <class 'SentimentAnalysis'>
print(f"情感: {result.sentiment}")
print(f"置信度: {result.confidence}")
print(f"关键词: {result.keywords}")

类型: <class '__main__.SentimentAnalysis'>
情感: positive
置信度: 0.98
关键词: ['实用', '学到', '强烈推荐']


## 2、高级特性

### 2.1 情况1：可选字段

举例：


In [49]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os


# 1、提供大模型
load_dotenv(override=True)
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    # temperature=1.5,
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL,
    extra_body={"enable_thinking": False},
    # max_tokens=10,
)

In [41]:

from pydantic import BaseModel, Field


class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")
    occupation: str = Field(description="职业")

structured_model = model.with_structured_output(Person)

result = structured_model.invoke("张三是一名软件工程师")

print(result)
print(type(result))

name='张三' age=30 occupation='软件工程师'
<class '__main__.Person'>


In [61]:

from pydantic import BaseModel, Field
from typing import Optional


class Person(BaseModel):
    """人物信息，禁止编造"""
    name: str = Field(description="姓名")
    age: Optional[int] = Field(description="年龄")
    occupation: str = Field(description="职业")

structured_model = model.with_structured_output(Person)

result = structured_model.invoke([
        ("system", "严格按照原文提取信息，原文不存在的内容返回null，绝对不允许自行猜测、编造任何内容；输出JSON格式"),
        ("human", "张三是一名出色软件工程师")
    ])

print(result)
print(type(result))
print(result.age)

name='张三' age=None occupation='软件工程师'
<class '__main__.Person'>
None


### 2.2 情况2：默认值

不同的模型供应商，对于此字段的支持是不同的。比如：closeai平台的gpt-5.4-mini就不支持此字段，而openrouter平台的gpt-5.4-mini就支持此字段

使用CloseAI平台的gpt模型：

In [63]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)
CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")

model_with_closeai = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=CLOSEAI_API_KEY,
    base_url=CLOSEAI_BASE_URL
)

使用OpenRouter平台的gpt模型：

In [62]:
from langchain_openrouter import ChatOpenRouter
from dotenv import load_dotenv
import os

load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE")

model_with_openrouter = ChatOpenRouter(
    model="openai/gpt-5.4-mini",
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
)

举例1：使用closeai

In [ ]:

from pydantic import BaseModel, Field


class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: int = Field(10, description="年龄")
    occupation: str = Field(description="职业")

structured_model = model_with_closeai.with_structured_output(Person)

result = structured_model.invoke([
        ("system", "你只能输出符合Person模型的JSON，不要思考过程、不要多余文字，禁止输出tool_use之类内部字段"),
        ("human", "张三是一名30岁的软件工程师")
])

print(result)
print(type(result))

作为对比举例2：使用openrouter

In [ ]:

from pydantic import BaseModel, Field


class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: int = Field(10, description="年龄")
    occupation: str = Field(description="职业")

structured_model = model_with_openrouter.with_structured_output(Person)

result = structured_model.invoke([
        ("system", "你只能输出符合Person模型的JSON，不要思考过程、不要多余文字，禁止输出tool_use之类内部字段"),
        ("human", "张三是一名30岁的软件工程师")
])

print(result)
print(type(result))

举例3：

In [ ]:
from typing import Optional
from pydantic import BaseModel, Field

class Product(BaseModel):
    """产品信息"""
    name: str = Field(description="产品名称")
    price: float = Field(description="价格")
    description: Optional[str] = Field(description="产品描述")
    stock: int = Field(default=100, description="库存")

# 测试
structured_llm = model_with_openrouter.with_structured_output(Product)
print("\n场景1：完整信息")
result1 = structured_llm.invoke("iPhone 17 售价 5999 元，最新款智能手机，库存 50台")
print(result1)

print("\n场景2：缺少描述和库存")
result2 = structured_llm.invoke("MacBook Pro 售价 12999 元")
print(result2)

### 2.3 情况3：枚举类型

举例1：方法1：

In [80]:
from enum import Enum
from typing import Optional
from pydantic import BaseModel, Field

# 定义你的优先级枚举类
class Priority(str, Enum):
    LOW = "低"
    MEDIUM = "中"
    HIGH = "高"

class CustomerInfo(BaseModel):
    """客户信息"""
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    urgency: Priority = Field(description="紧急程度")

# 测试
structured_llm = model_with_qwen.with_structured_output(CustomerInfo)
conversation = """
客服: 您好，请问有什么可以帮助您？
客户: 我是卢本伟，电话 138-1234-5678，希望半年内有新产品！不着急
客服: 好的，我帮您查一下
"""
result = structured_llm.invoke(f"从以下客服对话中提取客户信息：\n{conversation}")
print(result)

print("\n提取结果：")
print(f" 客户: {result.name}")
print(f" 电话: {result.phone}")
print(f" 邮箱: {result.email or '未提供'}")
print(f" 问题: {result.issue}")
print(f" 紧急程度: {result.urgency.value}")

name='卢本伟' phone='138-1234-5678' email=None issue='希望半年内有新产品' urgency=<Priority.LOW: '低'>

提取结果：
 客户: 卢本伟
 电话: 138-1234-5678
 邮箱: 未提供
 问题: 希望半年内有新产品
 紧急程度: 低


举例2：方法2：

In [82]:

from typing import Optional, Literal
from pydantic import BaseModel, Field


class CustomerInfo(BaseModel):
    """客户信息"""
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    urgency: Literal["低", "中", "高"] = Field(description="紧急程度")

# 测试
structured_llm = model_with_qwen.with_structured_output(CustomerInfo)
conversation = """
客服: 您好，请问有什么可以帮助您？
客户: 我是卢本伟，电话 138-1234-5678，希望半年内有新产品！不着急
客服: 好的，我帮您查一下
"""
result = structured_llm.invoke(f"从以下客服对话中提取客户信息：\n{conversation}")
print(result)

print("\n提取结果：")
print(f" 客户: {result.name}")
print(f" 电话: {result.phone}")
print(f" 邮箱: {result.email or '未提供'}")
print(f" 问题: {result.issue}")
print(f" 紧急程度: {result.urgency}")

name='卢本伟' phone='138-1234-5678' email=None issue='希望半年内有新产品' urgency='低'

提取结果：
 客户: 卢本伟
 电话: 138-1234-5678
 邮箱: 未提供
 问题: 希望半年内有新产品
 紧急程度: 低


### 2.4 情况4：列表提取

举例1：人物列表

In [83]:
from typing import List

class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: int  = Field(description="年龄")

class PersonList(BaseModel):
    """人物列表"""
    people : List[Person] # 多个Person的对象

structured_model = model_with_qwen.with_structured_output(PersonList)

result = structured_model.invoke("张三 30岁， 李四 49岁， 卢本伟，35岁")
print(result)


people=[Person(name='张三', age=30), Person(name='李四', age=49), Person(name='卢本伟', age=35)]


举例2：产品评论分析

In [88]:
class Review(BaseModel):
    """产品评论"""
    product: str
    rating: int = Field(description="评分 1-5")
    pros: List[str] = Field(description="优点列表")
    cons: List[str] = Field(description="缺点列表")

structured_model = model_with_qwen.with_structured_output(Review)

review = structured_model.invoke("iPhone 17 很棒！摄像头强大，手感好。但是价格贵，没有充电器。4分。")
print(review)
response = model_with_qwen.invoke(str(review))
print(response.content)

product='iPhone 17' rating=4 pros=['摄像头强大', '手感好'] cons=['价格贵', '没有充电器']
基于您提供的信息，以下是关于 **iPhone 17** 的产品评价摘要：

### 📱 产品：iPhone 17
**⭐ 综合评分：** 4.0 / 5.0

#### ✅ 优点 (Pros)
*   **摄像头强大**：影像系统表现出色，适合摄影爱好者。
*   **手感好**：机身设计符合人体工学，握持体验舒适。

#### ❌ 缺点 (Cons)
*   **价格贵**：售价较高，性价比可能不如部分竞品。
*   **没有充电器**：包装内不附带充电头，需额外购买或自备。

---
💡 **总结建议**：
如果您看重拍照效果和日常握持体验，且预算充足、家中已有兼容的充电设备，iPhone 17 是一个不错的选择；但对价格敏感或希望开箱即用的用户可能需要权衡一下额外成本。


举例3：文档信息提取

In [97]:
from typing import List
from pydantic import BaseModel, Field

class Invoice(BaseModel):
    """发票信息"""
    invoice_number: str = Field(description="发票号")
    date: str = Field(description="日期")
    total_amount: float = Field(description="总金额")
    items: List[str] = Field(description="商品")

# 测试
structured_llm = model.with_structured_output(Invoice)
invoice_text = """
发票号: INV-2024-001
日期: 2024-01-15
总金额: 1299.00
商品: MacBook Pro, AppleCare+
商品价格：MacBook Pro 899.00, AppleCare+ 400.00
"""
invoice = structured_llm.invoke(f"提取发票信息：{invoice_text}")
print(invoice)

invoice_number='INV-2024-001' date='2024-01-15' total_amount=1299.0 items=['{"name": "MacBook Pro", "price": 899.00}', '{"name": "AppleCare+", "price": 400.00}']


### 2.5 情况5：嵌套结构

举例1：

In [98]:

class Address(BaseModel):
    """地点描述"""
    city: str = Field(description="城市")
    district: str = Field(description="区域")

class Company(BaseModel):
    """公司信息"""
    name: str = Field(description="公司名称")
    address: Address = Field(description="公司所在地")

structured_model = model_with_qwen.with_structured_output(Company)

result = structured_model.invoke("阿里巴巴在杭州的滨江区")
print(result)

name='阿里巴巴杭州滨江园区' address=Address(city='杭州市', district='滨江区')


举例2：

In [107]:
from pydantic import BaseModel, Field
from typing import List

# 1. 定义嵌套的 Pydantic 模型
class Actor(BaseModel):
    """演员信息"""
    name: str = Field(description="演员姓名")
    role: str = Field(description="饰演的角色")

class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="上映年份")
    director: str = Field(description="导演")
    cast: List[Actor] = Field(description="演员列表")  # 嵌套Pydantic模型列表
    rating: float = Field(description="评分")

# 2. 初始化模型并绑定输出结构
structured_model = model_with_qwen.with_structured_output(Movie)

# 3. 调用模型，直接获取 Movie 实例
response = structured_model.invoke("请介绍电影《少林足球》")

# 4. 访问嵌套数据
print(f"电影名: {response.title}")
print(f"上映年份: {response.year}")
print(f"导演: {response.director}")
print(f"演员列表: {response.cast}")
print(f"评分: {response.rating}")

# 遍历嵌套的Actor对象
for actor in response.cast:
    print(f"演员：{actor.name}，角色：{actor.role}")

电影名: 《少林足球》 (Shaolin Soccer)
上映年份: 2001
导演: 周星驰
演员列表: [Actor(name='周星驰', role='五师兄 / 大力金刚腿'), Actor(name='赵薇', role='阿梅'), Actor(name='吴孟达', role='明锋 (前球星)'), Actor(name='谢贤', role='强雄 (反派老板)'), Actor(name='黄一飞', role='大师兄 / 铁头功'), Actor(name='莫美林', role='二师兄 / 旋风地堂腿'), Actor(name='田启文', role='三师兄 / 金钟罩铁布衫'), Actor(name='林子聪', role='四师兄 / 轻功水上漂')]
评分: 8.6
演员：周星驰，角色：五师兄 / 大力金刚腿
演员：赵薇，角色：阿梅
演员：吴孟达，角色：明锋 (前球星)
演员：谢贤，角色：强雄 (反派老板)
演员：黄一飞，角色：大师兄 / 铁头功
演员：莫美林，角色：二师兄 / 旋风地堂腿
演员：田启文，角色：三师兄 / 金钟罩铁布衫
演员：林子聪，角色：四师兄 / 轻功水上漂


举例3：

In [118]:
from pydantic import BaseModel, Field
from typing import List, Literal

class Aspect(BaseModel):
    """评论维度"""
    name: str = Field(description="维度名称，如：质量、价格、服务")
    score: int = Field(description="评分，1‑5")
    comment: str = Field(description="具体评价")

class ProductReview(BaseModel):
    """产品评论分析"""
    overall_sentiment: Literal["positive", "negative", "neutral"] = Field(description="整体情感")
    overall_score: int = Field(description="综合评分，1‑5")
    aspects: List[Aspect] = Field(description="各维度评价")
    summary: str = Field(description="一句话总结")

# 创建结构化模型
structured_model = model_with_qwen.with_structured_output(ProductReview)

# 测试
review_text = """
这款蓝牙耳机续航能力十分出色，长时间佩戴听歌完全够用。
音质通透立体，日常听音乐氛围感十足。
但是售价偏高，连续使用久了耳朵会酸胀。
店家解答问题耐心细致，发货速度很快。
总的来讲产品物有所值。
"""
result = structured_model.invoke(
    f"分析以下产品评论：\n{review_text}"
)

print(f"整体情感: {result.overall_sentiment}")
print(f"综合评分: {result.overall_score}/5")
print(f"\n各维度评价:")
for aspect in result.aspects:
    print(f" - {aspect.name}: {aspect.score}/5 - {aspect.comment}")
print(f"\n总结: {result.summary}")

整体情感: positive
综合评分: 4/5

各维度评价:
 - 续航能力: 5/5 - 十分出色，长时间佩戴听歌完全够用
 - 音质: 5/5 - 通透立体，日常听音乐氛围感十足
 - 价格: 2/5 - 售价偏高
 - 佩戴舒适度: 3/5 - 连续使用久了耳朵会酸胀
 - 客服服务: 5/5 - 解答问题耐心细致
 - 物流速度: 5/5 - 发货速度很快
 - 性价比: 4/5 - 总的来讲产品物有所值

总结: 用户对这款蓝牙耳机的核心功能（续航、音质）和服务体验（客服、物流）给予了高度评价。尽管存在价格偏高和长时间佩戴不适的缺点，但用户最终认为产品物有所值，整体评价为正面。


### 2.6 情况6：限制条件

参数	含义	  符号

ge	大于等于	≥

gt	大于	    ＞

le	小于等于	≤

lt	小于	    ＜

举例1：

In [135]:
from pydantic import ValidationError

class User(BaseModel):
    name: str = Field(description="姓名", min_length=2, max_length=50)
    age: int = Field(description="年龄", ge=0, le=150)
    email: str = Field(description="邮箱")

try:
    user1 = User(name="tom", age=20, email="tom@126.com")
    print(f"[ok]:{user1}")
except ValidationError as  e:
    print(f"[FAIL]: {e}]")

[ok]:name='tom' age=20 email='tom@126.com'


In [136]:
from pydantic import ValidationError

try:
    user2 = User(name="mike", age=190, email="tom@126.com")
    print(f"[ok]:{user2}")
except ValidationError as  e:
    print(f"[FAIL]: {e}]")

[FAIL]: 1 validation error for User
age
  Input should be less than or equal to 150 [type=less_than_equal, input_value=190, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/less_than_equal]


举例2：

In [140]:
from pydantic import BaseModel, Field

class Product(BaseModel):
    """产品信息（严格验证）"""
    name: str = Field(description="产品名称（字符串类型）", min_length=2)
    price: float = Field(description="价格，数字类型", gt=0)
    stock: int = Field(description="库存，整数类型", ge=0)

# 测试
structured_llm = model_with_qwen.with_structured_output(Product)
# response = structured_llm.invoke("华为mate 80 promax 价格是7999，当前库存100")
response = structured_llm.invoke("华为mate 80 promax 价格是-7999，当前库存-100")
print(response)

name='华为Mate 80 ProMax' price=7999.0 stock=100
